# Dataset Preparation

In [12]:
from pathlib import Path
import math
import time
import torch

In [13]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Project paths

In [14]:
BASE_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/LLM from Scratch"
)

DATA_DIR = BASE_DIR / "data"
RUNS_DIR = BASE_DIR / "runs" / "min-gpt"

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RUNS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## Device

In [15]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )

Device: cpu


## Reproducibility

In [16]:
SEED = 42

torch.manual_seed(SEED)

if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

## Dataset sources

In [17]:
BANGLA_DATASET = "barunsaha/bangla_sahitya"
ENGLISH_DATASET = "lucadiliello/bookcorpusopen"

print("\nProject directory:")
print(BASE_DIR)

print("\nData directory:")
print(DATA_DIR)

print("\nCheckpoint directory:")
print(RUNS_DIR)


Project directory:
/content/drive/MyDrive/Colab Notebooks/LLM from Scratch

Data directory:
/content/drive/MyDrive/Colab Notebooks/LLM from Scratch/data

Checkpoint directory:
/content/drive/MyDrive/Colab Notebooks/LLM from Scratch/runs/min-gpt


## Dataset Preparation — Hugging Face

In [18]:
!pip -q install datasets

from datasets import load_dataset

print("Hugging Face datasets ready.")

Hugging Face datasets ready.


In [19]:
bangla_ds = load_dataset(
    BANGLA_DATASET,
    split="train"
)

print("Bangla dataset loaded.")
print("Rows:", len(bangla_ds))
print("Columns:", bangla_ds.column_names)

README.md:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2593 [00:00<?, ? examples/s]

Bangla dataset loaded.
Rows: 2593
Columns: ['author', 'title', 'tag', 'text']


In [20]:
print("\nFirst row:")
print(bangla_ds[0])


First row:
{'author': 'অক্ষয়কুমার মৈত্রেয়', 'title': 'সিরাজদ্দৌলা', 'tag': 'prose', 'text': 'প্রথম পরিচ্ছেদ\n\nসেকালের সুখ দুঃখ।\n\nনবাব সিরাজদ্দৌলার নাম সকলের কাছেই চিরপরিচিত। তিনি অতি অল্পদিন মাত্র বাঙ্গালা, বিহার, উড়িষ্যার সিংহাসনে বসিয়াছিলেন; কিন্তু সেই অল্পদিনের মধ্যেই স্বদেশে এবং বিদেশে আপন নাম চিরস্মরণীয় করিয়া গিয়াছেন।\n\nইংরাজেরা একবার তাহাদের দেশের একজন হতভাগ্য নরপতিকে নর-বলি দিয়াছিল। ঘাতকের শাণিত কুঠার যখন সেই রাজমুণ্ড দ্বিখণ্ডিত করে, শোণিত লোলুপ জনসাধারণ তখন উন্মত্ত পিশাচের মত ভৈরব নৃত্যে করতালি দিয়া কিছুদিনের জন্য প্রজাতন্ত্র সংস্থাপন করিয়াছিল! কিন্তু তখন ও তাঁহাদের দেশের কুটীরে কুটীরে দুৰ্গে-দুর্গে, প্রাসাদে প্রাসাদে, কত কৃষক, কত সৈনিক, কত সন্ত্রান্ত পরিবার দীর্ঘনিঃশ্বাস ফেলিয়াছিল! বাঙ্গালী যখন ষড়যন্ত্র করিয়া সিরাজদ্দৌলাকে গৃহতাড়িত করে, মীরণের নৃশং আদেশে সিরাজমুণ্ড যখন দেহবিচুত হয়, দেশের রাজা প্রজা তখন সকলে মিলিয়া বিশ্বাসঘাতক মীর জাফরকে সিংহাসনে বসাইয়া তাঁহার কৃপা কটাক্ষের প্রতীক্ষায় করাজোড়ে দাড়াইয়াছিলেন;—সিরাজের শোচনীয় পরিণামে তাঁহার জন্য কেহই একবিন

In [21]:
bangla_texts = []

for row in bangla_ds:

    text = row.get("text")

    if text is None:
        continue

    text = str(text)

    if text.strip():
        bangla_texts.append(text)

In [22]:
bangla_text = "\n\n".join(
    bangla_texts
)

bangla_bytes = bangla_text.encode(
    "utf-8"
)

In [23]:
BANGLA_PATH = DATA_DIR / "bangla.txt"

BANGLA_PATH.write_bytes(
    bangla_bytes
)

96979450

In [24]:
print(f"Documents used  : {len(bangla_texts):,}")
print(f"Characters      : {len(bangla_text):,}")
print(f"UTF-8 bytes     : {len(bangla_bytes):,}")
print(f"Size            : {len(bangla_bytes) / (1024**2):.2f} MiB")
print(f"Saved to        : {BANGLA_PATH}")

Documents used  : 2,593
Characters      : 36,950,063
UTF-8 bytes     : 96,979,450
Size            : 92.49 MiB
Saved to        : /content/drive/MyDrive/Colab Notebooks/LLM from Scratch/data/bangla.txt


In [25]:
target_english_bytes = len(bangla_bytes)

print(
    f"Target English size: "
    f"{target_english_bytes / (1024**2):.2f} MiB"
)

Target English size: 92.49 MiB


In [26]:
english_stream = load_dataset(
    ENGLISH_DATASET,
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/400 [00:00<?, ?B/s]

In [27]:
english_texts = []
english_bytes_count = 0

for row in english_stream:

    text = row.get("text")

    if text is None:
        continue

    text = str(text)

    if not text.strip():
        continue

    text_bytes = len(
        text.encode("utf-8")
    )

    english_texts.append(text)
    english_bytes_count += text_bytes

    if english_bytes_count >= target_english_bytes:
        break

In [28]:
english_text = "\n\n".join(
    english_texts
)

english_bytes = english_text.encode(
    "utf-8"
)

In [29]:
ENGLISH_PATH = DATA_DIR / "english.txt"

ENGLISH_PATH.write_bytes(
    english_bytes
)

97258256

In [30]:
print(f"Documents selected : {len(english_texts):,}")
print(f"Characters         : {len(english_text):,}")
print(f"UTF-8 bytes        : {len(english_bytes):,}")
print(f"Size               : {len(english_bytes) / (1024**2):.2f} MiB")
print(f"Bangla target      : {target_english_bytes / (1024**2):.2f} MiB")
print(f"Saved to           : {ENGLISH_PATH}")

Documents selected : 278
Characters         : 96,223,181
UTF-8 bytes        : 97,258,256
Size               : 92.75 MiB
Bangla target      : 92.49 MiB
Saved to           : /content/drive/MyDrive/Colab Notebooks/LLM from Scratch/data/english.txt


In [31]:
import random

In [32]:
random.seed(SEED)

random.shuffle(bangla_texts)
random.shuffle(english_texts)

In [33]:
bangla_text = "\n\n".join(bangla_texts)
english_text = "\n\n".join(english_texts)

In [34]:
bangla_bytes = bangla_text.encode("utf-8")
english_bytes = english_text.encode("utf-8")
target_bytes = min(len(bangla_bytes), len(english_bytes))

print(f"Bangla : {len(bangla_bytes) / (1024**2):.2f} MiB")
print(f"English: {len(english_bytes) / (1024**2):.2f} MiB")
print(f"Target : {target_bytes / (1024**2):.2f} MiB")

Bangla : 92.49 MiB
English: 92.75 MiB
Target : 92.49 MiB


In [35]:
def truncate_utf8(text, max_bytes):
    data = text.encode("utf-8")
    if len(data) <= max_bytes:
        return text
    data = data[:max_bytes]
    # Remove a potentially incomplete UTF-8 character
    return data.decode("utf-8", errors="ignore")


bangla_text = truncate_utf8(bangla_text, target_bytes)
english_text = truncate_utf8(english_text, target_bytes)

In [36]:
separator = "\n\n" + "=" * 80 + "\n\n"
combined_text = bangla_text + separator + english_text

In [37]:
combined_bytes = combined_text.encode(
    "utf-8"
)

In [38]:
COMBINED_PATH = DATA_DIR / "combined.txt"

COMBINED_PATH.write_bytes(
    combined_bytes
)

193958984

In [39]:
print(f"Bangla bytes   : {len(bangla_text.encode('utf-8')):,}")
print(f"English bytes  : {len(english_text.encode('utf-8')):,}")
print(f"Separator bytes: {len(separator.encode('utf-8')):,}")
print(f"Total bytes    : {len(combined_bytes):,}")
print(f"Total size     : {len(combined_bytes) / (1024**2):.2f} MiB")
print(f"Total chars    : {len(combined_text):,}")
print(f"Saved to       : {COMBINED_PATH}")

Bangla bytes   : 96,979,450
English bytes  : 96,979,450
Separator bytes: 84
Total bytes    : 193,958,984
Total size     : 184.97 MiB
Total chars    : 132,894,527
Saved to       : /content/drive/MyDrive/Colab Notebooks/LLM from Scratch/data/combined.txt


In [40]:
# ============================================
# 0. Dataset Preparation — Verification
# ============================================

data = COMBINED_PATH.read_bytes()
decoded = data.decode("utf-8")
separator_pos = decoded.find(separator)

assert separator_pos != -1, "Language separator was not found."
assert len(data) == len(decoded.encode("utf-8")), "UTF-8 round-trip failed."

bangla_part = decoded[:separator_pos]
english_part = decoded[separator_pos + len(separator):]

print("============================================")
print("DATASET VERIFICATION")
print("============================================")
print(f"Total bytes   : {len(data):,}")
print(f"Bangla bytes  : {len(bangla_part.encode('utf-8')):,}")
print(f"English bytes : {len(english_part.encode('utf-8')):,}")
print(f"English/Bangla ratio: {len(english_part.encode('utf-8')) / len(bangla_part.encode('utf-8')):.4f}")

print("\nBangla sample:")
print(bangla_part[:300])

print("\nEnglish sample:")
print(english_part[:300])

print("\nLanguage boundary:")
print(repr(decoded[max(0, separator_pos - 80):separator_pos + len(separator) + 80]))

print("\n✓ Dataset verification passed.")

DATASET VERIFICATION
Total bytes   : 193,958,984
Bangla bytes  : 96,979,450
English bytes : 96,979,450
English/Bangla ratio: 1.0000

Bangla sample:
গগন-চটি

রাজশেখর বসু 


হাতিবাগানের দরজী আবুবকর মিঞা আর তার বউ রমজানী বিবি সন্ধ্যার সময় পশ্চিম আকাশে ঈদের চাঁদ দেখছিল। হঠাৎ একটা অদ্ভুত জিনিস রমজানীর নজরে পড়ল। সে তার স্বামীকে জিজ্ঞাসা করল, ও মিঞা, আসমানের মধ্যিখানে ছোট্ট কাটারির মতন জ্বলজ্বল করছে ওটা কি গো? আবুবকর অনেকক্ষণ ঠাহর করে বলল, কাটারি নয

English sample:


# 75 secrets.

## By Joaquin Emiliano

## Copyright 2018 Joaquin Emiliano

## Smashwords Edition

## Smashwords Edition, License Notes

## This ebook is licensed for your personal enjoyment only. This ebook may not be re-sold or given away to other people. If you would like to share this book with

Language boundary:
' বেবিলনে নিভে নতুন কলকাতাতে কবে\nক্রান্তি, সাগর, সূর্য জ্বলে অনাথ ইতিহাসের কলরবে।\n\n================================================================================\n\n\n\n# 75 secrets.\n\n## By Joaquin Emiliano

In [41]:
# Save session state
import torch

torch.save({
    "SEED": SEED,
    "device": str(device),
    "COMBINED_PATH": str(COMBINED_PATH),
    "BASE_DIR": str(BASE_DIR),
    "DATA_DIR": str(DATA_DIR),
    "RUNS_DIR": str(RUNS_DIR),
}, BASE_DIR / "session_state.pt")

print("Session state saved to Drive!")

Session state saved to Drive!


In [42]:
from pathlib import Path
import torch
from google.colab import drive
drive.mount("/content/drive")

state = torch.load("/content/drive/MyDrive/Colab Notebooks/LLM from Scratch/session_state.pt")

BASE_DIR    = Path(state["BASE_DIR"])
DATA_DIR    = Path(state["DATA_DIR"])
RUNS_DIR    = Path(state["RUNS_DIR"])
COMBINED_PATH = Path(state["COMBINED_PATH"])
SEED        = state["SEED"]
device      = torch.device(state["device"])

combined_text = COMBINED_PATH.read_text(encoding="utf-8")

print("Session restored!")
print(f"Device : {device}")
print(f"Chars  : {len(combined_text):,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Session restored!
Device : cpu
Chars  : 132,894,527


In [43]:
import os
import time
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

# Byte Tokenization

In [2]:
class ByteTokenizer:
    """Ultra-simple byte-level tokenizer.
    - encode(str) -> LongTensor [N]
    - decode(Tensor[int]) -> str
    - vocab_size = 256
    """
    def encode(self, s: str) -> torch.Tensor:
        return torch.tensor(list(s.encode("utf-8")), dtype=torch.long)

    def decode(self, ids) -> str:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        return bytes(ids).decode("utf-8", errors="ignore")

    @property
    def vocab_size(self) -> int:
        return 256

# Byte dataset

In [3]:
class ByteDataset:
    """Holds raw bytes of a text file and yields (x,y) blocks for LM."""
    def __init__(self, path: str, block_size: int = 256, split: float = 0.9):
        data = Path(path).read_bytes()
        data = torch.tensor(list(data), dtype=torch.long)

        n = int(len(data) * split)
        self.train = data[:n]
        self.val = data[n:]
        self.block_size = block_size

    def get_batch(self, which: str, batch_size: int, device: torch.device):
        buf = self.train if which == "train" else self.val

        assert len(buf) > self.block_size + 1, \
            "file too small for given block_size"

        ix = torch.randint(
            0,
            len(buf) - self.block_size - 1,
            (batch_size,),
        )

        x = torch.stack([
            buf[i:i + self.block_size]
            for i in ix
        ])

        y = torch.stack([
            buf[i + 1:i + 1 + self.block_size]
            for i in ix
        ])

        return x.to(device), y.to(device)

# Sampling utility

In [4]:
def top_k_top_p_filtering(
    logits: torch.Tensor,
    top_k: int | None = None,
    top_p: float | None = None,
):
    """Filter a distribution of logits using top-k and/or nucleus (top-p) filtering."""
    B, V = logits.shape
    filtered = logits.clone()

    if top_k is not None and top_k < V:
        topk_vals, _ = torch.topk(filtered, top_k, dim=-1)
        kth = topk_vals[:, -1].unsqueeze(-1)
        filtered[filtered < kth] = float("-inf")

    if top_p is not None and 0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(
            filtered,
            descending=True,
            dim=-1,
        )

        probs = torch.softmax(sorted_logits, dim=-1)
        cumsum = torch.cumsum(probs, dim=-1)

        mask = cumsum > top_p
        mask[..., 0] = False

        sorted_logits[mask] = float("-inf")

        filtered = torch.full_like(
            filtered,
            float("-inf"),
        )

        filtered.scatter_(
            1,
            sorted_idx,
            sorted_logits,
        )

    return filtered

# Causal self-attention

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.0):
        super().__init__()
        assert n_embd % n_head == 0

        self.n_head = n_head
        self.d_head = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.d_head)
        q, k, v = qkv.unbind(dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=None,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.proj(y)
        return y

# Feed-forward network

In [6]:
class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd: int,
        mult: int = 4,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, mult * n_embd),
            nn.GELU(),
            nn.Linear(mult * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Transformer Block

In [7]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, mult=4, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# GPT Model

In [8]:
class GPT(nn.Module):
    def __init__(
        self, vocab_size: int, block_size: int, n_layer: int = 4,
        n_head: int = 4, n_embd: int = 256, dropout: float = 0.0,
    ):
        super().__init__()
        self.block_size = block_size

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        assert T <= self.block_size

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self, idx: torch.Tensor, max_new_tokens: int = 200,
        temperature: float = 1.0, top_k: int | None = 50, top_p: float | None = None,
    ):
        self.eval()
        if idx.size(1) == 0:
            idx = torch.full((idx.size(0), 1), 10, dtype=torch.long, device=idx.device)

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

        return idx

# Model and training configuration

In [ ]:
VOCAB_SIZE, BLOCK_SIZE, BATCH_SIZE = 256, 256, 64
N_LAYER, N_HEAD, N_EMBD, DROPOUT = 8, 8, 512, 0.1
EPOCHS = 5
LR, WEIGHT_DECAY, GRAD_CLIP = 3e-4, 0.1, 1.0
EVAL_INTERVAL, EVAL_ITERS = 1, 50
SAMPLE_EVERY, SAMPLE_TOKENS = 1, 256
TEMPERATURE, TOP_K, TOP_P = 1.0, 50, None

CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RUN_DIR = RUNS_DIR / "min-gpt"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

tok = ByteTokenizer()
ds = ByteDataset(str(COMBINED_PATH), block_size=BLOCK_SIZE)

model = GPT(
    vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE,
    n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

steps_per_epoch = len(ds.train) // (BATCH_SIZE * BLOCK_SIZE)
total_steps = steps_per_epoch * EPOCHS

# Loss evaluation

In [9]:
def estimate_loss(
    model: GPT, ds: ByteDataset, eval_iters: int, batch_size: int, device: torch.device
) -> dict:
    model.eval()
    out = {}

    with torch.no_grad():
        for split in ["train", "val"]:
            losses = []
            for _ in range(eval_iters):
                xb, yb = ds.get_batch(split, batch_size, device)
                _, loss = model(xb, yb)
                losses.append(loss.item())
            out[split] = sum(losses) / len(losses)

    model.train()
    return out

# Epoch-based training with checkpoints

In [ ]:
best_val = float("inf")
start_epoch = 1

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0

    for step in range(steps_per_epoch):
        xb, yb = ds.get_batch("train", BATCH_SIZE, device)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            _, loss = model(xb, yb)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()

        if GRAD_CLIP > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(opt)
        scaler.update()
        train_loss_sum += loss.item()

        # ── live progress every 50 steps ──
        if (step + 1) % 50 == 0:
            avg = train_loss_sum / (step + 1)
            elapsed = time.time() - epoch_start
            print(
                f"  epoch {epoch}/{EPOCHS} | "
                f"step {step+1}/{steps_per_epoch} | "
                f"loss {loss.item():.4f} | "
                f"avg {avg:.4f} | "
                f"{elapsed:.0f}s",
                flush=True
            )

    train_loss = train_loss_sum / steps_per_epoch
    losses = estimate_loss(model, ds, EVAL_ITERS, BATCH_SIZE, device)
    val_loss = losses["val"]
    epoch_time = time.time() - epoch_start

    config = {
        "vocab_size": VOCAB_SIZE, "block_size": BLOCK_SIZE, "n_layer": N_LAYER,
        "n_head": N_HEAD, "n_embd": N_EMBD, "dropout": DROPOUT,
    }
    ckpt_payload = {
        "model": model.state_dict(), "optimizer": opt.state_dict(),
        "epoch": epoch, "val_loss": val_loss, "config": config,
    }

    epoch_ckpt = CHECKPOINT_DIR / f"epoch_{epoch}.pt"
    torch.save(ckpt_payload, epoch_ckpt)

    if val_loss < best_val:
        best_val = val_loss
        best_ckpt = RUN_DIR / "model_best.pt"
        torch.save(ckpt_payload, best_ckpt)

    print(
        f"\n✓ epoch {epoch:2d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_loss:.4f} | "
        f"time {epoch_time:.1f}s\n"
    )

    if SAMPLE_EVERY > 0 and epoch % SAMPLE_EVERY == 0:
        start = torch.randint(low=0, high=len(ds.train) - BLOCK_SIZE - 1, size=(1,)).item()
        seed = ds.train[start:start + BLOCK_SIZE].unsqueeze(0).to(device)
        out = model.generate(seed, max_new_tokens=SAMPLE_TOKENS, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)
        txt = tok.decode(out[0].cpu())
        print("\n================ SAMPLE ================\n"
              + txt[-(BLOCK_SIZE + SAMPLE_TOKENS):]
              + "\n=======================================\n")

# To resume from epoch 3
Make sure to run all class definition cells


```
ByteTokenizer
ByteDataset
top_k_top_p_filtering
CausalSelfAttention
FeedForward
Block
GPT
estimate_loss
```



In [44]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR       = Path("/content/drive/MyDrive/Colab Notebooks/LLM from Scratch")
DATA_DIR       = BASE_DIR / "data"
RUNS_DIR       = BASE_DIR / "runs"
COMBINED_PATH  = DATA_DIR / "combined.txt"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RUN_DIR        = RUNS_DIR / "min-gpt"

SEED         = 42
VOCAB_SIZE   = 256
BLOCK_SIZE   = 256
BATCH_SIZE   = 64
N_LAYER      = 8
N_HEAD       = 8
N_EMBD       = 512
DROPOUT      = 0.1
LR           = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP    = 1.0
EPOCHS       = 5
EVAL_ITERS   = 50
SAMPLE_EVERY = 1
SAMPLE_TOKENS = 256
TEMPERATURE  = 1.0
TOP_K        = 50
TOP_P        = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu


In [ ]:
import torch

# Resume from epoch 2 checkpoint
ckpt = torch.load(CHECKPOINT_DIR / "epoch_2.pt", map_location=device)

config = ckpt["config"]
model = GPT(**config).to(device)
model.load_state_dict(ckpt["model"])

opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
opt.load_state_dict(ckpt["optimizer"])

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

start_epoch = 3  # resume from epoch 3
print(f"Resumed from epoch 2, val loss was {ckpt['val_loss']:.4f}")

In [45]:
ds = ByteDataset(str(COMBINED_PATH), block_size=BLOCK_SIZE)
tok = ByteTokenizer()
steps_per_epoch = len(ds.train) // BATCH_SIZE // BLOCK_SIZE
print(f"Steps/epoch: {steps_per_epoch}")

Steps/epoch: 10654


In [46]:
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

In [ ]:
best_val = ckpt["val_loss"]  # 1.2797 from epoch 2
# start_epoch is already 3 from resume cell

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0

    for step in range(steps_per_epoch):
        xb, yb = ds.get_batch("train", BATCH_SIZE, device)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            _, loss = model(xb, yb)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()

        if GRAD_CLIP > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(opt)
        scaler.update()
        train_loss_sum += loss.item()

        if (step + 1) % 50 == 0:
            avg = train_loss_sum / (step + 1)
            elapsed = time.time() - epoch_start
            print(
                f"  epoch {epoch}/{EPOCHS} | "
                f"step {step+1}/{steps_per_epoch} | "
                f"loss {loss.item():.4f} | "
                f"avg {avg:.4f} | "
                f"{elapsed:.0f}s",
                flush=True
            )

    train_loss = train_loss_sum / steps_per_epoch
    losses = estimate_loss(model, ds, EVAL_ITERS, BATCH_SIZE, device)
    val_loss = losses["val"]
    epoch_time = time.time() - epoch_start

    config = {
        "vocab_size": VOCAB_SIZE, "block_size": BLOCK_SIZE, "n_layer": N_LAYER,
        "n_head": N_HEAD, "n_embd": N_EMBD, "dropout": DROPOUT,
    }
    ckpt_payload = {
        "model": model.state_dict(), "optimizer": opt.state_dict(),
        "epoch": epoch, "val_loss": val_loss, "config": config,
    }

    epoch_ckpt = CHECKPOINT_DIR / f"epoch_{epoch}.pt"
    torch.save(ckpt_payload, epoch_ckpt)

    if val_loss < best_val:
        best_val = val_loss
        best_ckpt = RUN_DIR / "model_best.pt"
        torch.save(ckpt_payload, best_ckpt)

    print(
        f"\n✓ epoch {epoch:2d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_loss:.4f} | "
        f"time {epoch_time:.1f}s\n"
    )

    if SAMPLE_EVERY > 0 and epoch % SAMPLE_EVERY == 0:
        start = torch.randint(low=0, high=len(ds.train) - BLOCK_SIZE - 1, size=(1,)).item()
        seed = ds.train[start:start + BLOCK_SIZE].unsqueeze(0).to(device)
        out = model.generate(seed, max_new_tokens=SAMPLE_TOKENS, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)
        txt = tok.decode(out[0].cpu())
        print("\n================ SAMPLE ================\n"
              + txt[-(BLOCK_SIZE + SAMPLE_TOKENS):]
              + "\n=======================================\n")

In [ ]:
ckpt = torch.load(CHECKPOINT_DIR / "epoch_4.pt", map_location=device)
config = ckpt["config"]
model = GPT(**config).to(device)
model.load_state_dict(ckpt["model"])
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
opt.load_state_dict(ckpt["optimizer"])
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
start_epoch = 5
best_val = ckpt["val_loss"]
print(f"Resumed from epoch 4, val loss was {ckpt['val_loss']:.4f}")

In [ ]:
best_val = ckpt["val_loss"]  # 1.2800 from epoch 4

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    train_loss_sum = 0.0

    for step in range(steps_per_epoch):
        xb, yb = ds.get_batch("train", BATCH_SIZE, device)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            _, loss = model(xb, yb)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()

        if GRAD_CLIP > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(opt)
        scaler.update()
        train_loss_sum += loss.item()

        if (step + 1) % 50 == 0:
            avg = train_loss_sum / (step + 1)
            elapsed = time.time() - epoch_start
            print(
                f"  epoch {epoch}/{EPOCHS} | "
                f"step {step+1}/{steps_per_epoch} | "
                f"loss {loss.item():.4f} | "
                f"avg {avg:.4f} | "
                f"{elapsed:.0f}s",
                flush=True
            )

        # ── sample every 100 steps ──
        if (step + 1) % 100 == 0:
            model.eval()
            sample_start = torch.randint(low=0, high=len(ds.train) - BLOCK_SIZE - 1, size=(1,)).item()
            seed = ds.train[sample_start:sample_start + BLOCK_SIZE].unsqueeze(0).to(device)
            with torch.no_grad():
                out = model.generate(seed, max_new_tokens=SAMPLE_TOKENS, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)
            txt = tok.decode(out[0].cpu())
            print("\n================ SAMPLE ================\n"
                  + txt[-(BLOCK_SIZE + SAMPLE_TOKENS):]
                  + "\n=======================================\n",
                  flush=True)
            model.train()

    train_loss = train_loss_sum / steps_per_epoch
    losses = estimate_loss(model, ds, EVAL_ITERS, BATCH_SIZE, device)
    val_loss = losses["val"]
    epoch_time = time.time() - epoch_start

    config = {
        "vocab_size": VOCAB_SIZE, "block_size": BLOCK_SIZE, "n_layer": N_LAYER,
        "n_head": N_HEAD, "n_embd": N_EMBD, "dropout": DROPOUT,
    }
    ckpt_payload = {
        "model": model.state_dict(), "optimizer": opt.state_dict(),
        "epoch": epoch, "val_loss": val_loss, "config": config,
    }

    epoch_ckpt = CHECKPOINT_DIR / f"epoch_{epoch}.pt"
    torch.save(ckpt_payload, epoch_ckpt)

    if val_loss < best_val:
        best_val = val_loss
        best_ckpt = RUN_DIR / "model_best.pt"
        torch.save(ckpt_payload, best_ckpt)

    print(
        f"\n✓ epoch {epoch:2d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_loss:.4f} | "
        f"time {epoch_time:.1f}s\n"
    )

In [47]:
ckpt = torch.load(CHECKPOINT_DIR / "epoch_5.pt", map_location=device)
config = ckpt["config"]
model = GPT(**config).to(device)
model.load_state_dict(ckpt["model"])
print(f"Loaded epoch 5, val loss: {ckpt['val_loss']:.4f}")

Loaded epoch 5, val loss: 1.2479


In [ ]:
model.eval()

def chat(prompt, max_new_tokens=200, temperature=1.0, top_k=50):
    prompt_bytes = tok.encode(prompt)
    idx = prompt_bytes.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)

    return tok.decode(out[0].cpu())

# Test it
while True:
    prompt = input("\nYou: ")
    if prompt.lower() in ["exit", "quit"]:
        break
    response = chat(prompt)
    print(f"\nModel: {response}")

In [49]:
script = '''
import torch
from pathlib import Path
import sys

# ── paste your class definitions here ──
# ByteTokenizer, GPT, Block, CausalSelfAttention, FeedForward, top_k_top_p_filtering

CHECKPOINT = Path("/content/drive/MyDrive/Colab Notebooks/LLM from Scratch/checkpoints/epoch_5.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CHECKPOINT, map_location=device)
model = GPT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

tok = ByteTokenizer()

def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    idx = tok.encode(prompt).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tok.decode(out[0].cpu())[len(prompt):]

print("=" * 50)
print("  Bilingual LLM — Text Completion CLI")
print("  :quit to exit")
print("=" * 50)

temperature = 0.8
max_new_tokens = 200

while True:
    try:
        prompt = input("\\nPrompt: ")
    except (EOFError, KeyboardInterrupt):
        print("\\nBye!")
        sys.exit(0)

    if not prompt.strip():
        continue
    if prompt.strip() == ":quit":
        print("Bye!")
        break
    if prompt.strip().startswith(":temp"):
        temperature = float(prompt.strip().split()[1])
        print(f"Temperature: {temperature}")
        continue
    if prompt.strip().startswith(":tokens"):
        max_new_tokens = int(prompt.strip().split()[1])
        print(f"Max tokens: {max_new_tokens}")
        continue

    print("\\nOutput:", end=" ", flush=True)
    print(generate(prompt, max_new_tokens, temperature))
'''

with open("/content/cli.py", "w") as f:
    f.write(script)

print("Saved to /content/cli.py")

Saved to /content/cli.py


In [51]:
full_script = """
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import sys

class ByteTokenizer:
    def encode(self, s):
        return torch.tensor(list(s.encode("utf-8")), dtype=torch.long)
    def decode(self, ids):
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        return bytes(ids).decode("utf-8", errors="ignore")
    @property
    def vocab_size(self):
        return 256

def top_k_top_p_filtering(logits, top_k=None, top_p=None):
    B, V = logits.shape
    filtered = logits.clone()
    if top_k is not None and top_k < V:
        topk_vals, _ = torch.topk(filtered, top_k, dim=-1)
        kth = topk_vals[:, -1].unsqueeze(-1)
        filtered[filtered < kth] = float("-inf")
    if top_p is not None and 0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(filtered, descending=True, dim=-1)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumsum = torch.cumsum(probs, dim=-1)
        mask = cumsum > top_p
        mask[..., 0] = False
        sorted_logits[mask] = float("-inf")
        filtered = torch.full_like(filtered, float("-inf"))
        filtered.scatter_(1, sorted_idx, sorted_logits)
    return filtered

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        self.n_head = n_head
        self.d_head = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
            dropout_p=self.dropout.p if self.training else 0.0, is_causal=True)
        y = y.transpose(1,2).contiguous().view(B, T, C)
        return self.proj(y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, mult * n_embd), nn.GELU(),
            nn.Linear(mult * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout=dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=4, n_head=4, n_embd=256, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks: x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=1.0, top_k=None, top_p=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

CHECKPOINT = Path("/content/drive/MyDrive/Colab Notebooks/LLM from Scratch/checkpoints/epoch_5.pt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT, map_location=device)
model = GPT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
tok = ByteTokenizer()

def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    idx = tok.encode(prompt).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tok.decode(out[0].cpu())[len(prompt):]

print("=" * 50)
print("  Bilingual LLM — Text Completion CLI")
print("  :quit | :temp 0.8 | :tokens 200")
print("=" * 50)

temperature = 0.8
max_new_tokens = 200

while True:
    try:
        prompt = input("\\nPrompt: ")
    except (EOFError, KeyboardInterrupt):
        print("\\nBye!")
        sys.exit(0)
    if not prompt.strip(): continue
    if prompt.strip() == ":quit": break
    if prompt.strip().startswith(":temp"):
        temperature = float(prompt.strip().split()[1])
        print(f"Temperature: {temperature}")
        continue
    if prompt.strip().startswith(":tokens"):
        max_new_tokens = int(prompt.strip().split()[1])
        print(f"Max tokens: {max_new_tokens}")
        continue
    print("\\nOutput:", end=" ", flush=True)
    print(generate(prompt, max_new_tokens, temperature))
"""

with open("/content/cli.py", "w") as f:
    f.write(full_script)

print("Done! Run in terminal: python /content/cli.py")

Done! Run in terminal: python /content/cli.py
